# Exploratory Data Analysis for Forex Data

This notebook explores historical forex price data to understand distributions, patterns, and statistical properties.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Project imports
import sys
sys.path.insert(0, '..')
from env.preprocessor import FeatureProcessor, normalize_ohlcv

## Load Data

In [ ]:
# Load sample data
data_path = Path('../data/raw/EURUSD.npy')
csv_path = Path('../data/raw/EURUSD.csv')

if csv_path.exists():
    df = pd.read_csv(csv_path)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp')
else:
    data = np.load(data_path)
    df = pd.DataFrame(data, columns=['open', 'high', 'low', 'close', 'volume'])

print(f'Data shape: {df.shape}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
df.head()

## Basic Statistics

In [ ]:
print('=== Price Statistics ===')
print(df[['open', 'high', 'low', 'close']].describe())

print('\n=== Volume Statistics ===')
print(df['volume'].describe())

## Price Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Close price histogram
axes[0, 0].hist(df['close'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Close Price Distribution')
axes[0, 0].set_xlabel('Price')
axes[0, 0].set_ylabel('Frequency')

# Volume histogram
axes[0, 1].hist(df['volume'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_title('Volume Distribution')
axes[0, 1].set_xlabel('Volume')
axes[0, 1].set_ylabel('Frequency')

# Price over time
axes[1, 0].plot(df.index, df['close'], linewidth=0.5)
axes[1, 0].set_title('Close Price Over Time')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Price')
plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=45)

# Volume over time
axes[1, 1].plot(df.index, df['volume'], linewidth=0.5, color='orange')
axes[1, 1].set_title('Volume Over Time')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Volume')
plt.setp(axes[1, 1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## Returns Analysis

In [ ]:
# Calculate returns
df['returns'] = df['close'].pct_change()
df['log_returns'] = np.log(df['close'] / df['close'].shift(1))

print('=== Returns Statistics ===')
print(df['returns'].describe())

print('\n=== Log Returns Statistics ===')
print(df['log_returns'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Returns histogram with normal fit
axes[0].hist(df['log_returns'].dropna(), bins=100, density=True, alpha=0.7, edgecolor='black')
x = np.linspace(df['log_returns'].min(), df['log_returns'].max(), 100)
from scipy import stats
mu, sigma = stats.norm.fit(df['log_returns'].dropna())
axes[0].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label=f'Normal fit\nμ={mu:.6f}\nσ={sigma:.6f}')
axes[0].set_title('Log Returns Distribution')
axes[0].set_xlabel('Log Return')
axes[0].set_ylabel('Density')
axes[0].legend()

# Q-Q plot
stats.probplot(df['log_returns'].dropna(), dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Log Returns')

plt.tight_layout()
plt.show()

## Volatility Analysis

In [ ]:
# Calculate rolling volatility
df['rolling_vol_24'] = df['log_returns'].rolling(24).std() * np.sqrt(24 * 365)  # Annualized
df['rolling_vol_168'] = df['log_returns'].rolling(168).std() * np.sqrt(168 * 365)  # Weekly

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Rolling volatility
axes[0].plot(df.index, df['rolling_vol_24'], label='24h Volatility', linewidth=0.5)
axes[0].plot(df.index, df['rolling_vol_168'], label='168h Volatility', linewidth=0.5)
axes[0].set_title('Rolling Volatility (Annualized)')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Volatility')
axes[0].legend()
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

# Volatility histogram
axes[1].hist(df['rolling_vol_24'].dropna(), bins=50, alpha=0.7, edgecolor='black')
axes[1].set_title('24h Volatility Distribution')
axes[1].set_xlabel('Volatility')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Candlestick Analysis

In [ ]:
# Calculate candlestick features
df['body_size'] = abs(df['close'] - df['open'])
df['upper_shadow'] = df['high'] - df[['open', 'close']].max(axis=1)
df['lower_shadow'] = df[['open', 'close']].min(axis=1) - df['low']
df['total_range'] = df['high'] - df['low']
df['body_ratio'] = df['body_size'] / df['total_range'].replace(0, np.nan)

print('=== Candlestick Statistics ===')
print(df[['body_size', 'upper_shadow', 'lower_shadow', 'total_range', 'body_ratio']].describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Body size distribution
axes[0, 0].hist(df['body_size'] * 10000, bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Body Size Distribution (pips)')
axes[0, 0].set_xlabel('Body Size (pips)')
axes[0, 0].set_ylabel('Frequency')

# Body ratio distribution
axes[0, 1].hist(df['body_ratio'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_title('Body Ratio Distribution')
axes[0, 1].set_xlabel('Body / Total Range')
axes[0, 1].set_ylabel('Frequency')

# Upper vs Lower shadow scatter
axes[1, 0].scatter(df['upper_shadow'] * 10000, df['lower_shadow'] * 10000, alpha=0.1, s=1)
axes[1, 0].plot([0, 50], [0, 50], 'r--', linewidth=2)
axes[1, 0].set_title('Upper vs Lower Shadow (pips)')
axes[1, 0].set_xlabel('Upper Shadow (pips)')
axes[1, 0].set_ylabel('Lower Shadow (pips)')
axes[1, 0].set_xlim(0, 50)
axes[1, 0].set_ylim(0, 50)

# Total range over time
axes[1, 1].plot(df.index, df['total_range'] * 10000, linewidth=0.5)
axes[1, 1].set_title('Total Range Over Time (pips)')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Range (pips)')
plt.setp(axes[1, 1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## Correlation Analysis

In [ ]:
# Calculate correlations
corr_cols = ['open', 'high', 'low', 'close', 'volume', 'returns', 'body_size', 'total_range']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Summary

In [ ]:
print('=== Data Summary ===')
print(f'Total bars: {len(df):,}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
print(f'\nPrice range: {df["low"].min():.5f} - {df["high"].max():.5f}')
print(f'Average daily range: {df["total_range"].mean() * 10000:.2f} pips')
print(f'\nMean return: {df["returns"].mean() * 100:.4f}%')
print(f'Return std: {df["returns"].std() * 100:.4f}%')
print(f'Max daily gain: {df["returns"].max() * 100:.4f}%')
print(f'Max daily loss: {df["returns"].min() * 100:.4f}%')
print(f'\nSkewness: {df["log_returns"].skew():.4f}')
print(f'Kurtosis: {df["log_returns"].kurtosis():.4f}')
print(f'\nAnnualized volatility: {df["log_returns"].std() * np.sqrt(24 * 365) * 100:.2f}%')